# Testing NVIDIA NeMo Switchyard for Model Routing
### A wealth-management agent workload

This notebook adapts the LangChain benchmark
[*How many of your agent's calls actually need a frontier model?*](https://www.langchain.com/blog/switchyard-agent-routing-benchmark)
to a **wealth-management** agent workload.

**The question:** on our agent's turns, how many actually need a frontier model, and what
does routing the rest to a smaller model cost in accuracy?

**What we'll do**
1. Compute the **cost tradeoff formula** *first* — it can rule routing out before you build anything.
2. Run **Path A — the Switchyard server in escalation mode** (reproduces the benchmark's headline setup).
3. Run **Path B — the in-process `SwitchyardRoutingMiddleware`** with Deep Agents.
4. Inspect routing decisions and tally where calls (and money) went.

> **Switchyard is pre-alpha.** APIs change fast; the server is a demo server, not for
> production. Pin versions. All sample data here is synthetic and not investment advice.

## 0. Two paths, and why they differ

This is the single most important thing to understand.

| | **Path A — Server, escalation** | **Path B — Middleware, stage_router** |
|---|---|---|
| Reproduces benchmark numbers? | **Yes** (74% cheaper / ~6 pts) | No (different algorithm) |
| How it decides | A **judge model** votes on each weak-tier turn; 2 consecutive "off track" verdicts escalate the task | Reads **tool-call / tool-result signals** in the message history; no judge by default |
| Added latency | ~700ms/turn (extra judge call) until escalation | ~0 |
| Deployment | Standalone Rust proxy; point `base_url` at it | In-process; no separate service |
| Config | `routes.toml` (`type="llm_classifier"`, `mode="escalation"`) | `algorithms.stage_router(...)` |

The benchmark states plainly that the middleware **will not reproduce the escalation
numbers directly**. Use the server to reproduce; use the middleware for the cleanest
in-process integration.

## 1. Setup

In [8]:
# Load environment (.env). Copy .env.example -> .env and fill in your keys first.
from dotenv import load_dotenv
import os

load_dotenv()

for key in ["NVIDIA_API_KEY", "ANTHROPIC_API_KEY", "GOOGLE_API_KEY", "OPENROUTER_API_KEY"]:
    print(f"{key:20s} {'set' if os.getenv(key) else 'MISSING'}")

NVIDIA_API_KEY       MISSING
ANTHROPIC_API_KEY    MISSING
GOOGLE_API_KEY       MISSING
OPENROUTER_API_KEY   MISSING


In [9]:
# Make src/ importable
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

from tasks import TASKS, as_messages
from cost_formula import minimum_offload

print(f"Loaded {len(TASKS)} wealth-management tasks.")

Loaded 9 wealth-management tasks.


## 2. Run the cost formula FIRST

```
minimum offload = judge cost / (expensive cost - cheap cost)
```

The judge is a **fixed tax** on every run (it reads every weak-tier turn until a task
escalates). So the question is never *"is my cheap model good enough?"* — it's *"is the
price gap between my two models wide enough to pay for the judge?"*

If the required offload exceeds 100%, routing **cannot pay** against the strong model
alone unless you shrink the judge or self-host the cheap model.

Plug in **your desk's** approved trio's per-run costs below.

In [10]:
# Benchmark's own numbers (sanity check): judge $0.64/run, gap $10.73 -> ~5.9% needed.
benchmark = minimum_offload(judge_cost=0.64, expensive_cost=0.72 + 10.73, cheap_cost=0.72)
print("Benchmark check:", benchmark.summary())
print()

# --- EDIT THESE for your own weak/strong/judge trio (per-run cost estimates) ---
your_judge_cost = 0.64      # fixed judge cost per run
your_expensive_cost = 11.45 # strong model, per run
your_cheap_cost = 0.72      # weak model, per run

econ = minimum_offload(
    judge_cost=your_judge_cost,
    expensive_cost=your_expensive_cost,
    cheap_cost=your_cheap_cost,
)
print("Your workload:", econ.summary())
if not econ.feasible:
    print("\n>>> Routing is ruled out on cost. Shrink the judge or self-host the cheap model.")

Benchmark check: Minimum offload = 6.0%: route at least this share of turns to the cheap model for routing to beat the strong model alone.

Your workload: Minimum offload = 6.0%: route at least this share of turns to the cheap model for routing to beat the strong model alone.


### Reading the result

- **Feasible + low bar (e.g. < 10%):** routing very likely pays — offloading even a modest
  share of turns beats the strong model alone. (The benchmark cleared its 5.9% bar by ~16x.)
- **Feasible + high bar (e.g. 60-100%):** marginal. You need most turns to be genuinely easy.
- **Infeasible (> 100%):** the two models are too close in price. No judge config fixes this;
  self-hosting the cheap model (near-zero inference cost) is the only lever.

The formula tells you whether good routing decisions would be *worth paying for*. It cannot
tell you whether the router will *make* good decisions on your traffic — that's what Paths A/B measure.

## 3. Path A - Switchyard server, escalation mode (reproduces the benchmark)

Switchyard's core is a **Rust proxy**. You run it as a standalone server, point your agent's
`base_url` at it, and it translates between OpenAI Chat / Anthropic Messages / Gemini formats
while routing per your `routes.toml`.

### 3a. Install and start the server (run in a terminal, not the notebook)

```bash
# 1. Install Rust + Cargo:  https://rust-lang.org/tools/install/
# 2. Install the server binary:
cargo install --locked switchyard-server

# 3. Validate the config (dry run) and start it:
switchyard-server --config ../config/routes.toml --dry-run
switchyard-server --config ../config/routes.toml --host 127.0.0.1 --port 4000
```

`config/routes.toml` in this repo is already set to **escalation mode** with a weak/strong/judge
trio and `confirmations = 2` (two strikes -> escalate). Swap the model IDs for your approved trio.

In [11]:
# Health check the running server (start it in a terminal first - see the cell above).
import requests

SERVER_URL = "http://127.0.0.1:4000"
try:
    r = requests.get(f"{SERVER_URL}/health", timeout=3)
    print("Server health:", r.status_code, r.text[:200])
    SERVER_UP = r.ok
except Exception as e:
    SERVER_UP = False
    print(f"Server not reachable at {SERVER_URL}. Start it in a terminal first.\n  {e}")

Server health: 200 {"status":"ok"}


### 3b. Point a Deep Agent at the server

The agent speaks **OpenAI Chat** to Switchyard; Switchyard picks weak/strong per the
escalation route and translates to each provider's native format. We use an OpenAI-compatible
chat client with `base_url` set to the proxy.

In [12]:
# Requires: pip install langchain-openai deepagents
# The `model` here is the route id from routes.toml ("switchyard"), not a provider model name.
from langchain_openai import ChatOpenAI

if SERVER_UP:
    routed_model = ChatOpenAI(
        model="switchyard",              # the route id in routes.toml
        base_url=f"{SERVER_URL}/v1",
        api_key="not-used-by-proxy",     # the proxy holds the real provider keys
        # NOTE: temperature intentionally omitted -- some reasoning models 400 on it.
    )
    print("Routed model client ready (talks to the Switchyard proxy).")
else:
    routed_model = None
    print("Skipping - start the server first.")

Routed model client ready (talks to the Switchyard proxy).


In [14]:
from deepagents import create_deep_agent

if routed_model is not None:
    agent = create_deep_agent(model=routed_model)

    # Run one HARD task - this is the kind of turn we'd expect to escalate.
    hard_task = next(t for t in TASKS if t.difficulty == "hard")
    result = agent.invoke({"messages": as_messages(hard_task)})
    print("TASK:", hard_task.id, f"({hard_task.difficulty})")
    print(result["messages"][-1].content[:800])
else:
    print("Skipping agent run - server not up.")

TASK: wm-007 (hard)
Below is a reasonable default sequence, assuming the client has no high-interest debt, the 401(k) offers an employer match, and the 529 is owned by the parent. The exact order depends heavily on their age, tax bracket, retirement spending need, and current balances.

## 1. Establish the liquidity reserve first

Maintain at least 12 months of household expenses in highly liquid, low-volatility assets—such as a high-yield savings account, money-market fund, or short-term Treasury bills.

Because retirement is only eight years away, it may also be appropriate to hold:

- The next one to three years of expected retirement withdrawals in cash or short-term bonds; and
- The child’s college funds in a separate, increasingly conservative portfolio.

The emergency reserve should not be invested agg


### 3c. Run the full sample set through the server and tally escalations

In the benchmark, **7% of calls** went to the frontier model but carried **68% of the bill**;
the judge itself took **21.2%** of routed spend. Here we run our small set and count how many
tasks escalated. (For real numbers, use a larger eval set with ground-truth scoring.)

In [ ]:
import time

def run_all_through_server(agent, tasks):
    rows = []
    for t in tasks:
        t0 = time.time()
        res = agent.invoke({"messages": as_messages(t)})
        rows.append({
            "id": t.id,
            "difficulty": t.difficulty,
            "category": t.category,
            "seconds": round(time.time() - t0, 2),
            "answer_chars": len(res["messages"][-1].content),
        })
    return rows

if routed_model is not None:
    rows = run_all_through_server(agent, TASKS)
    for r in rows:
        print(r)
    print("\nNOTE: escalation happens inside the proxy. Check the server's Prometheus")
    print("metrics / logs to see per-target call counts, or use LangSmith tracing.")
else:
    print("Skipping - server not up.")

> **Where to read routing decisions on Path A:** the proxy exposes Prometheus metrics
> (requests, tokens, latency, routing overhead per target). Scrape `/metrics` on the
> server, or enable LangSmith tracing to see per-turn model selection.

## 4. Path B - In-process middleware (`SwitchyardRoutingMiddleware`)

No separate service. Routing happens inside the agent process, and every routed `AIMessage`
carries its decision trace in `response_metadata["switchyard"]`.

The benchmark's numbers come from Path A. Path B's `stage_router` is a **different
algorithm** - it keys off tool-call / tool-result signals rather than a judge's verdict -
so it won't reproduce the escalation numbers, but it adds ~0 latency and suits tool-dense agents.

### 4a. Install from source (middleware is not on PyPI)

```bash
# Python 3.12+ required.
git clone https://github.com/NVIDIA-NeMo/Switchyard.git
python -m pip install -e ./Switchyard

git clone https://github.com/langchain-ai/langchain-nvidia.git
python -m pip install -e "./langchain-nvidia/libs/switchyard[openrouter]"
```

The middleware routes models through **OpenRouter**, so set `OPENROUTER_API_KEY` and confirm
your account can access both configured models.

In [ ]:
# Argument order matters: capable target FIRST, efficient target SECOND.
try:
    from langchain_openrouter import ChatOpenRouter
    from switchyard.libsy import LlmTarget, algorithms
    from langchain_nvidia_switchyard import LangChainLlmClient, SwitchyardRoutingMiddleware

    efficient_model = ChatOpenRouter(model="nvidia/nemotron-3.5-lightning-30b-a3b")
    capable_model = ChatOpenRouter(model="anthropic/claude-opus-4.8")

    router = algorithms.stage_router(
        LlmTarget("capable", LangChainLlmClient(capable_model)),
        LlmTarget("efficient", LangChainLlmClient(efficient_model)),
        picker="efficient_first",
        confidence_threshold=0.5,
        recent_window=3,
    )
    middleware = SwitchyardRoutingMiddleware(router)
    MIDDLEWARE_READY = True
    print("Switchyard middleware ready (stage_router, efficient_first).")
except ImportError as e:
    MIDDLEWARE_READY = False
    print(f"Middleware not installed - see 4a. Import error:\n  {e}")

In [ ]:
from deepagents import create_deep_agent

if MIDDLEWARE_READY:
    # Deep Agents needs a base model; the middleware substitutes its own choice per call.
    # Reuse the efficient target to avoid constructing an unused third model.
    agent_mw = create_deep_agent(model=efficient_model, middleware=[middleware])

    # ainvoke is canonical in notebooks.
    task = next(t for t in TASKS if t.difficulty == "easy")
    result_mw = await agent_mw.ainvoke({"messages": as_messages(task)})
    print("TASK:", task.id, f"({task.difficulty})")
    print(result_mw["messages"][-1].content[:600])
else:
    print("Skipping - middleware not installed.")

### 4b. Inspect routing decisions

`response_metadata["switchyard"]` carries `selected_model` (final pick) and `decisions`
(full ordered trace when an algorithm decides more than once).

In [ ]:
from langchain.messages import AIMessage

if MIDDLEWARE_READY:
    msg = next(
        m for m in reversed(result_mw["messages"]) if isinstance(m, AIMessage)
    )
    routing = msg.response_metadata.get("switchyard", {})
    print("selected_model:", routing.get("selected_model"))
    print("decisions:     ", routing.get("decisions"))
else:
    print("Skipping - middleware not installed.")

In [ ]:
from collections import Counter

async def run_all_middleware(agent, tasks):
    picks = Counter()
    rows = []
    for t in tasks:
        res = await agent.ainvoke({"messages": as_messages(t)})
        msg = next(m for m in reversed(res["messages"]) if isinstance(m, AIMessage))
        sel = msg.response_metadata.get("switchyard", {}).get("selected_model", "unknown")
        picks[sel] += 1
        rows.append({"id": t.id, "difficulty": t.difficulty, "selected_model": sel})
    return rows, picks

if MIDDLEWARE_READY:
    rows_mw, picks = await run_all_middleware(agent_mw, TASKS)
    for r in rows_mw:
        print(r)
    print("\nSelected-model tally:", dict(picks))
else:
    print("Skipping - middleware not installed.")

## 5. Routing all 3 models through LangChain's LLM Gateway

So far the router calls each provider directly. This section puts **LangChain's LLM
Gateway** underneath the router so every model call — weak, strong, and judge — flows
through one governed endpoint.

### Two layers, two jobs

| Layer | Job |
|---|---|
| **Switchyard** | The **router** — picks weak / strong / judge per turn |
| **LLM Gateway** | The **execution + governance** layer — makes the actual call, applies spend / rate-limit / PII policy, and writes a **LangSmith trace** per call |

```
agent ──> Switchyard (routes) ──> LLM Gateway (governs + calls) ──> Anthropic / Gemini / Nemotron
```

### Why do this
- **One key.** Authenticate with a single **LangSmith API key**; provider keys live in
  workspace **Provider Secrets**, not in your config or `.env`.
- **One request format.** The Gateway speaks OpenAI Chat / Anthropic Messages and
  translates to each provider's native format — so all three models share one client type.
- **Governance on every routed call.** Spend caps, rate limits, and PII redaction apply
  *regardless of which tier the router picked*.
- **Tracing.** Every call (including the judge) shows up as a LangSmith trace.

### One-time admin setup (LangSmith UI)
1. Enable LLM Gateway for the workspace.
2. Add **Provider Secrets** for Anthropic, Google (Gemini), and your Nemotron endpoint
   (self-hosted OSS -> add it as a [custom provider](https://docs.langchain.com/langsmith/llm-gateway-custom-providers)),
   or use Gateway Credits where available.
3. Grant your workspace-scoped LangSmith API key access.

Then set `LANGSMITH_API_KEY` in `.env`. Model IDs become **provider-prefixed** Gateway
IDs: `anthropic/claude-opus-4-8`, `google/gemini-3.1-flash-lite`, and a custom-provider
ID for Nemotron.

### 5a. Path B (middleware) through the Gateway

This is the simplest wiring: build each LangChain model with `base_url` = the Gateway and
`api_key` = your LangSmith key, then hand them to `LangChainLlmClient`. `src/gateway_models.py`
does exactly this.

In [ ]:
# Build the weak / strong / judge trio, all pointed at the LLM Gateway.
from gateway_models import build_trio, gateway_chat, GATEWAY_BASE_URL, WEAK_ID, STRONG_ID, JUDGE_ID

print("Gateway base URL:", GATEWAY_BASE_URL)
print("weak :", WEAK_ID)
print("strong:", STRONG_ID)
print("judge:", JUDGE_ID)

# Each is a standard ChatOpenAI whose calls route through the Gateway.
weak_model, strong_model, judge_model = build_trio()
print("\nBuilt 3 Gateway-backed models.")

In [ ]:
# Wire the Gateway-backed models into the Switchyard middleware (Path B).
# Same stage_router as section 4 — only the model construction changed.
try:
    from switchyard.libsy import LlmTarget, algorithms
    from langchain_nvidia_switchyard import LangChainLlmClient, SwitchyardRoutingMiddleware
    from deepagents import create_deep_agent

    gw_router = algorithms.stage_router(
        LlmTarget("capable", LangChainLlmClient(strong_model)),   # capable FIRST
        LlmTarget("efficient", LangChainLlmClient(weak_model)),   # efficient SECOND
        picker="efficient_first",
        confidence_threshold=0.5,
        recent_window=3,
    )
    gw_agent = create_deep_agent(
        model=weak_model,
        middleware=[SwitchyardRoutingMiddleware(gw_router)],
    )
    print("Deep agent ready: Switchyard routing -> LLM Gateway execution.")
    GW_READY = True
except ImportError as e:
    GW_READY = False
    print(f"Switchyard middleware not installed - see section 4a. Import error:\n  {e}")

In [ ]:
# Run one task. The routing decision is Switchyard's; the call is executed by the Gateway
# (and is now visible as a LangSmith trace with spend/policy applied).
from langchain.messages import AIMessage

if GW_READY:
    task = next(t for t in TASKS if t.difficulty == "easy")
    res = await gw_agent.ainvoke({"messages": as_messages(task)})
    msg = next(m for m in reversed(res["messages"]) if isinstance(m, AIMessage))
    print("TASK:", task.id)
    print("selected_model:", msg.response_metadata.get("switchyard", {}).get("selected_model"))
    print(msg.content[:500])
    print("\nCheck smith.langchain.com -> this workspace for the traced Gateway call.")
else:
    print("Skipping - middleware not installed.")

### 5b. Path A (server) through the Gateway

For the server path, use **`config/routes.gateway.toml`** instead of `routes.toml`. It
points every `llm_client` at the Gateway base URL, authenticates with `LANGSMITH_API_KEY`,
and uses provider-prefixed Gateway model IDs. The escalation route itself is unchanged.

```bash
switchyard-server --config ../config/routes.gateway.toml --dry-run
switchyard-server --config ../config/routes.gateway.toml --host 127.0.0.1 --port 4000
```

Your agent still points at the Switchyard proxy (`http://127.0.0.1:4000`) exactly as in
section 3 — the difference is that Switchyard now forwards each upstream call to the
Gateway rather than straight to Anthropic / Google / NVIDIA.

**Note on double-tracing:** with the server path, calls are traced by the Gateway. If you
*also* enable LangSmith tracing on the agent, you'll see both the agent-side and
Gateway-side spans — useful, but be aware of it.

### Which layer does what (recap)

- **Switchyard owns the routing decision** — weak vs strong vs judge, escalation logic,
  `confirmations`. The Gateway does **not** make routing decisions between your tiers.
- **The Gateway owns execution + governance** — the actual provider call, spend/rate/PII
  policy, and tracing, applied uniformly no matter which tier was chosen.
- **Cost formula still applies unchanged.** The Gateway may change your *per-call prices*
  (e.g. Gateway Credits vs BYO provider), so re-run section 2 with the Gateway's effective
  costs. The judge is still a fixed tax; the weak/strong price gap still decides feasibility.

### 5c. Content-based routing that escalates hard tasks (capability mode)

**Why does everything still go to the weak model?** Sections 4 and 5a use
`stage_router`, which is **signal-driven** — it routes on tool-call / tool-result / error
signals in the message history. Our sample tasks are plain Q&A with **no tools**, so
stage_router has no signals to read and keeps every turn on the efficient tier. Making a
prompt *sound* harder doesn't help: stage_router doesn't read content difficulty.

Two algorithms **do** read request content and can escalate a hard task with no tools:

| Mode | Route `type` / `mode` | How it decides |
|---|---|---|
| **Escalation** (section 5b) | `llm_classifier` + `mode="escalation"` | Starts on weak; escalates only after the judge sees repeated bad turns. On a single easy turn it never escalates. |
| **Capability** (this section) | `llm_classifier` + `mode="capability"` | Judge estimates `p_solve` (can the weak model complete THIS task?) **up front**; routes to strong when `p_solve` < threshold. |

**Capability mode is the one that sends a genuinely hard task straight to the bigger
model** — which is what you want for a "low cost, high complexity escalates" demo.

#### Run capability mode (server path)

Capability mode is a route configuration, so run it through the Switchyard **server** using
`config/routes.gateway.capability.toml` (weak=luna, strong=terra, classifier/judge=sol,
all via the Gateway):

```bash
switchyard-server --config ../config/routes.gateway.capability.toml --dry-run
switchyard-server --config ../config/routes.gateway.capability.toml --host 127.0.0.1 --port 4000
```

The judge returns a structured verdict per request:
- `p_solve` — probability the weak model completes the task
- `capability_boundary` — `supported` / `uncertain` / `unsupported` / `unmatched`

Route to weak when `p_solve >= base_threshold` (0.5 here); otherwise to strong. Harder
boundaries add `threshold_step`, so `unsupported` tasks need a higher `p_solve` to stay on
weak — i.e. they escalate. **Raise `base_threshold` to push more traffic to the strong model.**

In [ ]:
# Send the easy task and the VERY HARD task through the capability route and compare.
# The easy one should stay on weak (luna); the very-hard one should escalate to strong (terra).
import requests

CAP_URL = "http://127.0.0.1:4000"  # server started with routes.gateway.capability.toml

def route_once(prompt: str):
    r = requests.post(
        f"{CAP_URL}/v1/chat/completions",
        json={"model": "switchyard", "messages": [{"role": "user", "content": prompt}]},
        timeout=120,
    )
    r.raise_for_status()
    body = r.json()
    # The upstream model that actually served the request is echoed in the response `model`.
    return body.get("model", "?"), body["choices"][0]["message"]["content"][:300]

try:
    easy = next(t for t in TASKS if t.difficulty == "easy")
    hard = next(t for t in TASKS if t.difficulty == "very_hard")
    for label, t in [("EASY", easy), ("VERY HARD", hard)]:
        served_by, preview = route_once(t.prompt)
        print(f"[{label}] task={t.id}  served_by={served_by}")
        print("  ", preview.replace(chr(10), " ")[:200], "...\n")
    print("Expect: EASY -> weak (luna), VERY HARD -> strong (terra).")
    print("Routing is model-dependent output, not a fixed result -- rerun to see variation.")
except Exception as e:
    print(f"Capability server not reachable at {CAP_URL}. Start it first (see cell above).\n  {e}")

In [ ]:
# DIAGNOSTIC: is the server actually running capability mode, and what does the judge say?
# Run this against the server on port 4000 to find out WHY everything is going to luna.
import requests, json as _json

CAP_URL = "http://127.0.0.1:4000"

# 1) Full raw response for the very-hard task, including headers. Switchyard often exposes
#    the chosen route/target and decision info in response HEADERS (x-switchyard-*).
hard = next(t for t in TASKS if t.difficulty == "very_hard")
resp = requests.post(
    f"{CAP_URL}/v1/chat/completions",
    json={"model": "switchyard", "messages": [{"role": "user", "content": hard.prompt}]},
    timeout=180,
)
print("HTTP", resp.status_code)
print("\n--- switchyard/routing response headers ---")
sy_headers = {k: v for k, v in resp.headers.items() if "switchyard" in k.lower() or "route" in k.lower() or "model" in k.lower()}
print(_json.dumps(sy_headers, indent=2) if sy_headers else "(no switchyard/route headers exposed)")
print("ALL headers:", list(resp.headers.keys()))

body = resp.json()
print("\nserved model (body):", body.get("model"))

# 2) If served model == weak on the very-hard task, the judge scored p_solve too high,
#    OR the server is NOT in capability mode (e.g. still the escalation config).
#    Restart the server with the capability config to be sure:
#      switchyard-server --config ../config/routes.gateway.capability.toml --host 127.0.0.1 --port 4000
print("\nIf this shows luna on the VERY HARD task, either:")
print("  (a) the server is not running routes.gateway.capability.toml (restart it), or")
print("  (b) the judge returned a high p_solve -> raise base_threshold (0.5 -> 0.7) and restart.")


#### Tuning knobs (so you can force / suppress escalation)

- **`base_threshold`** (0-1): the `p_solve` bar to stay on weak. **Higher = more escalation.**
  If your very-hard task still lands on weak, raise it (e.g. 0.6-0.7).
- **`threshold_step`**: extra bar added for harder capability boundaries. Raises escalation
  for `uncertain` / `unsupported` tasks specifically.
- **`recent_turn_window`**: how much conversation the judge sees (unset = opening task +
  latest user message). **`session_affinity` / `message_hash_fallback`**: reuse one verdict
  across a session instead of re-judging every request (cuts judge cost).
- **Fail-open:** an unparseable verdict or judge error routes to **strong**. So if the judge
  (sol) returns empty/garbled JSON, you'll see everything escalate — check the judge output.

The cost formula from section 2 still governs whether this pays: the judge is a fixed tax,
and the weak->strong price gap must be wide enough to cover it.

## 6. Takeaways

1. **Run the formula first.** If the weak/strong price gap is too narrow, routing can't pay -
   no eval needed. Self-hosting the cheap model is the main lever when it's marginal.
2. **The value comes from moving traffic *away* from the frontier model**, not toward it. The
   savings are in sending easy turns to the cheap model; escalation is insurance for the hard ones.
3. **The judge is a real cost** (21.2% of routed spend in the benchmark). A cheaper judge, or
   one that skips clearly-fine turns, comes straight off the bill.
4. **Two paths, two purposes.** Server escalation reproduces the numbers; middleware
   stage_router is the clean in-process option for tool-dense agents.
5. **This is one workload.** The benchmark suite was saturated (~8 pts between 30B and
   frontier). Score against *your* eval set with ground truth before acting.

### Next steps for a real evaluation
- Replace `src/tasks.py` with a labeled eval set (ground-truth answers -> real accuracy).
- Try `confirmations = 1` vs `2` on the server route and measure the escalation-rate / cost swing.
- Try a cheaper judge model.
- Compare a self-hosted weak model (near-zero inference cost) to widen the price gap.